# DAAAnalyzer_2.0.AnalyzeABRs.ipynb
Created by: JFranco | Created On: 27 NOV 2024 | Last Env: QISHA | Last Run Date: 15 DEC 2024

This Python notebook will read in all DPOAE files for a specific DAA Run as specified by the user. It will make a lineplot for all availble files and label them by category. 

In [ ]:
import numpy as np
import seaborn as sns
import pandas as pd
import glob
import os
import xlrd
import math
import matplotlib.pyplot as plt
from IPython.display import clear_output
import warnings
warnings.filterwarnings("ignore")

In [ ]:
#                *** WHAT TO ANALYZE // WHERE TO GET/STORE **
# Key identifiers
prepID = 'CMI_005'
daaRun = '1'
daaType = 'Pre-Noise Exposure'
fnMD = prepID+'.Metadata.AnimalInfo.csv'

# Directories 
#   existing ones 
dirMain = '/Users/joyfranco/Dropbox (Partners HealthCare)/JF_Shared/Data/CFTS/'
dirPrep = dirMain+prepID+'/'
dirData = dirPrep+'RawData/'
dirMD = dirPrep+'Metadata/'

#   ones that need to be made
#   DPOAE folder is #2
dirSV = dirPrep+'Results/'
dirDFs = dirSV+"1_CompiledMeasurements/"
dirSjABRPlts = dirSV+"3_ABRPlotsBySubject/"
dirGpABRPlts = dirSV+"4_ABRPlotsByGroup/"
dirSjWOPlts = dirSV+"5_Wave1PlotsBySubject/"
dirGpWOPlts = dirSV+"6_Wave1PlotsByGroup/"

In [ ]:
#      *** MAKE SUBFOLDERS AS NECESSARY**
# Create directory for storing spreadsheets and summary plots if they don't exist
if not os.path.exists(dirSV): os.mkdir(dirSV)
if not os.path.exists(dirDFs): os.mkdir(dirDFs)
if not os.path.exists(dirSjABRPlts): os.mkdir(dirSjABRPlts)
if not os.path.exists(dirGpWOPlts): os.mkdir(dirGpABRPlts)
if not os.path.exists(dirSjWOPlts): os.mkdir(dirSjWOPlts)
if not os.path.exists(dirGpWOPlts): os.mkdir(dirGpWOPlts)

In [ ]:
#    *** LOAD MD SHEETS **
# Sample metadata sheet that include animal and condition info
dfMD = pd.read_csv(dirMD+fnMD)
dfMD.reset_index(inplace=True)
dfMD

In [ ]:
#      *** GENERATE A LIST OF DAA FOLDERS THAT WILL BE ANALYZED **
# Get a list of all the folders that match the prep
allData = os.listdir(dirData)
dataList = []
for datum in allData:
    if ('.'+daaRun) in datum:
        dataList.append(datum)
dataList

In [ ]:
#     *** GENERATE A DATAFRAME OF ALL ABR DATA FOR THIS RUN ***
dfAll = pd.DataFrame()
for daa in dataList:
    # Setup the path to the file
    abrFold = dirData+daa+'/'
    # Get a list of available ABR files
    allFiles = os.listdir(abrFold)
    anaFiles = []
    dfFAll = pd.DataFrame()
    for file in allFiles:
        # Only consider files that are analyzed 
        if ("analyzed.txt") in file:
            # Get information about the recording subject
            anID = daa[8:-2]
            # Get information about this animal from the MD sheet
            inMD = dfMD.index[dfMD['AnimalID']==anID].tolist()[0]
            inj = dfMD.loc[inMD]['InjGroup']
            sex = dfMD.loc[inMD]['Sex']
            # Get the base filename
            strFile = str(file)
            inPost = strFile.index("analyzed.txt")
            basename = strFile[0:(inPost-1)]
            # Open the analyzed file to get the ABR frequency for this file
            with open(abrFold+basename+"-analyzed.txt", "r") as f:
                anaABR = f.read()
                inPre = anaABR.index("(kHz):")
                inPost = anaABR.index("Threshold estimation:")
                freq = anaABR[(inPre+6):(inPost-2)]
            # Read in the analyzed ABR file 
            dfAA = pd.read_csv(abrFold+basename+"-analyzed.txt", sep="\t", skiprows=6)
            # Read in the filtered ABR file 
            dfFA = pd.read_csv(abrFold+basename+"-filtered.txt", delimiter='\t', header=0,index_col="Time (ms)")
            # Get the number of dB levels for the file
            for level in dfFA.columns:
                # Get the WI amplitudes for this level
                inLV = dfAA.index[dfAA[dfAA.columns[0]]==float(level[0:-3])].tolist()[0]
                waveOne = dfAA.loc[inLV]['P1 Amplitude']
                data = {'AnimalID':anID,"CMIID":prepID,"Sex":sex,"InjGroup":inj,
                        "AnimalFile":anID+"."+basename, "File":basename,"DAA_Run":daaRun,
                       "Freq_kHz":float(freq),"Level":level[0:-5], "Time_ms":dfFA.index,"Amp_uV":dfFA[level].values,"P1_Amplitude_uV":waveOne}
                df = pd.DataFrame(data)
                df['AnimalID'] = anID
                df['CMIID'] = prepID
                df['DAA_Run'] = daaRun
                dfFAll = pd.concat([dfFAll,df])
                dfAll = pd.concat([dfAll,df])

# Save the main df that has ABR data for all files in this run 
dfAll.reset_index(inplace=True)
dfAll.to_csv(dirDFs+prepID+'.AllABRVals.'+daaRun+'.csv') 

In [ ]:
# Get list of available frequencies to analyze


In [52]:
# *** IDENTIFY THRESHOLDS FOR EACH ABR FILE ***
# Setup df to store threshold information for each file
dfTh = pd.DataFrame()
# Get list of all files to eval
allFiles = dfAll["AnimalFile"].unique()
for file in dfAll["AnimalFile"].unique():
    # Get metadata for this file
    inFile = dfAll.index[dfAll['AnimalFile']==file].tolist()[0]
    anID = dfAll.loc[inFile]['AnimalID']
    cmiID = dfAll.loc[inFile]['CMIID']
    sex = dfAll.loc[inFile]['Sex']
    injG = dfAll.loc[inFile]['InjGroup']
    fName = dfAll.loc[inFile]['File']
    daaRun = dfAll.loc[inFile]['DAA_Run']
    freq = dfAll.loc[inFile]['Freq_kHz']
    
    """ 
    # Setup graph to show user to get threshold for this file
    g = sns.FacetGrid(dfAll[dfAll['AnimalFile']==file], row="Level", aspect=4, height=1, 
                  row_order=np.sort(dfAll["Level"].unique())[::-1],hue="Level")
    g.map(sns.lineplot, "Time_ms", "Amp_uV")
    # Define and use a simple function to label the plot in axes coordinates
    def label(x, color, label):
        ax = plt.gca()
        ax.text(0, .5, label, fontweight="bold", color=color,
                ha="right", va="center", transform=ax.transAxes)
    g.map(label, "Level")
    # Adjust plot aesthetics
    g.set(xlim=(0, 9), ylim=(-2.5, 2.5))
    # Set the subplots to overlap
    g.figure.subplots_adjust(hspace=-.1)
    # Remove axes details that don't play well with overlap
    g.set_titles("")
    g.set(yticks=[], ylabel="")
    g.set(xlabel="Time (msec)")
    g.despine(bottom=True, left=True)
    plt.show()
    # Get threshold from user
    th = input("What is the threshold for this file?\n"+
                "Be sure to enter it as shown on the plot: ")
    clear_output(wait=True)
    """

     # Temporary manual input for testing purposes
    th = "35"

    # Get threshold & 80 dB wave I amplitudes for this file
    #    Find the index for the th
    dfSS = dfAll[dfAll['AnimalFile']==file]
    inTh = dfSS.index[dfSS['Level']==th].tolist()[0]
    wOneTh = dfSS.loc[inTh]['P1_Amplitude_uV']
    inEdB = dfSS.index[dfSS['Level']=="80"].tolist()[0]
    wOneEdB = dfSS.loc[inEdB]['P1_Amplitude_uV']

    # Format information for this file into df
    data = {'AnimalID':anID,"CMIID":prepID,"Sex":sex,"InjGroup":injG, "File":fName,"DAA_Run":daaRun,
            "Freq_kHz":freq, "Th_dB":th, "W1uV_Th":wOneTh, "W1uV_80dB":wOneEdB}
    df = pd.DataFrame(data, index=[0])

    # Add info to df for all 
    dfTh = pd.concat([dfTh,df])

# Save the threshold df 
dfTh.reset_index(inplace=True)
dfTh.to_csv(dirDFs+prepID+'.ABRThresholdData.'+daaRun+'.csv') 

In [ ]:
# *** MAKE SUBJECT SUMMARY PLOTS ***
# Iterate through all of the subjects
for daa in dataList:
    # Get information about the recording subject
    anID = daa[8:-2]
    daaRun = daa[-1:]
    
    # *** Create & save the ABR Waveform plots for all available ABR files for this animal ***
    g = sns.FacetGrid(dfAll[dfAll["AnimalID"]==anID], col="Freq_kHz",row="Level", aspect=4, height=1, 
                      col_order=np.sort(dfAll["Freq_kHz"].unique()), row_order=np.sort(dfFAll["Level"].unique())[::-1],hue="Level", margin_titles=True)
    g.map(sns.lineplot, "Time_ms", "Amp_uV")
    g.tight_layout()
    g.refline(y=0, linewidth=.5, linestyle="--", color=None, clip_on=False)
    g.refline(x=2, linewidth=.5, linestyle="--", color=None, clip_on=False)
    # Define and use a simple function to label the plot in axes coordinates
    def label(x, color, label):
        ax = plt.gca()
        ax.text(0, .5, label+" dB", fontweight="bold", color=color,
                ha="right", va="center", transform=ax.transAxes)
    g.map(label, "Level")
    # Adjust plot aesthetics
    g.set(xlim=(0, 9), ylim=(-2.5, 2.5))
    g.figure.subplots_adjust(hspace=-.2)
    g.figure.suptitle(prepID+" | "+anID+" | "+daaType, fontsize=12)
    g.fig.subplots_adjust(top=.95)
    g.set_titles(col_template="{col_name}", row_template=" ")
    g.set(yticks=[], ylabel="")
    g.set(xlabel="Time (msec)")
    g.despine(bottom=True, left=True)
    # Save the figure
    plt.savefig(dirSjABRPlts+prepID+'.'+anID+'.'+daaRun+'.ABRs.png',  bbox_inches='tight', dpi = 300,format = 'png')
    plt.close()

    # *** Create & save the Wave 1 amplitudes for all available ABR files for this animal ***
    ax = sns.lineplot(dfAll[dfAll["AnimalID"]==anID], x="Level", y="P1_Amplitude_uV",hue="Freq_kHz")
    ax.set_title(prepID+" "+anID+" | "+daaType)
    ax.set(xlabel="Tone Level (dB SPL)")
    ax.set(ylabel="Wave One Amplitude (uV)")
    plt.savefig(dirSjWOPlts+prepID+'.'+anID+'.'+daaRun+'.Wave1Amp.png',  bbox_inches='tight', dpi = 300,format = 'png')
    plt.close()

In [ ]:
# *** MAKE GROUP SUMMARY PLOTS ***
# Make group summary ABR plot
g = sns.FacetGrid(dfAll[dfAll["Freq_kHz"]==8], col="InjGroup",row="Level", aspect=4, height=1, 
                  col_order=dfAll[dfAll["Freq_kHz"]==8]["InjGroup"].unique(), row_order=np.sort(dfFAll["Level"].unique())[::-1],hue="Level")
g.map(sns.lineplot, "Time_ms", "Amp_uV")
g.tight_layout()
g.refline(y=0, linewidth=.5, linestyle="--", color=None, clip_on=False)
g.refline(x=2, linewidth=.5, linestyle="--", color=None, clip_on=False)
# Define and use a simple function to label the plot in axes coordinates
def label(x, color, label):
    ax = plt.gca()
    ax.text(0, .5, label, fontweight="bold", color=color,
            ha="right", va="center", transform=ax.transAxes)
g.map(label, "Level")
# Adjust plot aesthetics
g.set(xlim=(0, 9), ylim=(-2.5, 2.5))
g.figure.subplots_adjust(hspace=-.2)
g.figure.suptitle(prepID+" | "+daaType+"\n8 kHz", fontsize=12)
g.fig.subplots_adjust(top=.95)
g.set_titles(col_template="{col_name}", row_template=" ")
g.set(yticks=[], ylabel="")
g.set(xlabel="Time (msec)")
g.despine(bottom=True, left=True)
# Save the figure
plt.savefig(dirGpABRPlts+prepID+'.'+daaRun+'.ABRs.png',  bbox_inches='tight', dpi = 300,format = 'png')
plt.close()

In [55]:
# Make summary wave 1 plot
ax = sns.lineplot(dfAll[dfAll["Freq_kHz"]==32], x="Level", y="P1_Amplitude_uV",hue="InjGroup")
ax.set_title(prepID+" | "+daaType)
ax.set(xlabel="Tone Level (dB SPL)")
ax.set(ylabel="Wave One Amplitude (uV)")
plt.savefig(dirGpWOPlts+prepID+'.'+daaRun+'.Wave1Amp.png',  bbox_inches='tight', dpi = 300,format = 'png')
plt.close()